In [1]:
from typing import Any
import torch
import torch.nn as nn
import torch.nn.functional as f
from torchvision.models import resnet50, swin_t, swin_s, swin_b, swin_v2_t, swin_v2_s, swin_v2_b
from torchvision.models import ResNet50_Weights, Swin_T_Weights, Swin_S_Weights, Swin_B_Weights, Swin_V2_T_Weights, Swin_V2_S_Weights, Swin_V2_B_Weights
from enum import Enum
from vision_studio.models import BaseModel
from vision_studio.dataset import ImageNetClassificationDataset
from vision_studio.data_loader import SimpleDataLoader
from vision_studio.trainer import WandbTrainer
from vision_studio.augmentation import Resize
from torch.optim import SGD, AdamW 

/home/local/.conda/envs/vision-studio/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
class ResNet50(BaseModel):
    def __init__(self, num_classes: int, weights: ResNet50_Weights | None):
        super().__init__()
        self.num_classes = num_classes
        
        self.model = resnet50(weights=weights)
        
        model_in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(model_in_features, num_classes)

    def forward(self, inputs, targets=None):
        logits = self.model(inputs)
        
        # Using dim=1 because dim 0 is the batch size
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(logits, dim=1)
        
        return {
            'logits': logits,
            'probs': probs,
            'labels': labels,
        }

    def compute_loss(self, outputs, targets):
        logits = outputs["logits"]
        labels = targets["label"]
        loss = f.cross_entropy(logits, labels)
        return {"loss": loss}

In [3]:



class SwinModelSelect(Enum):
    SWIN_T = 'SWIN_T'
    SWIN_S = 'SWIN_S'
    SWIN_B = 'SWIN_B'
    SWIN_V2_T = 'SWIN_V2_T'
    SWIN_V2_S = 'SWIN_V2_S'
    SWIN_V2_B = 'SWIN_V2_B'

class SwinTransformer(BaseModel):
    def __init__(self, model_select: SwinModelSelect, num_classes: int, weights: Any):
        super().__init__()
        self.num_classes = num_classes
        
        if model_select == SwinModelSelect.SWIN_T:  
            if weights is not None and not isinstance(weights, Swin_T_Weights):
                raise ValueError(f"Expected weights to be of type Swin_T_Weights, got {type(weights)}")
            self.model = swin_t(weights=weights)
        elif model_select == SwinModelSelect.SWIN_S:
            if weights is not None and not isinstance(weights, Swin_S_Weights):
                raise ValueError(f"Expected weights to be of type Swin_S_Weights, got {type(weights)}")
            self.model = swin_s(weights=weights)
        elif model_select == SwinModelSelect.SWIN_B:
            if weights is not None and not isinstance(weights, Swin_B_Weights):
                raise ValueError(f"Expected weights to be of type Swin_B_Weights, got {type(weights)}")
            self.model = swin_b(weights=weights)
        elif model_select == SwinModelSelect.SWIN_V2_T:
            if weights is not None and not isinstance(weights, Swin_V2_T_Weights):
                raise ValueError(f"Expected weights to be of type Swin_V2_T_Weights, got {type(weights)}")
            self.model = swin_v2_t(weights=weights)
        elif model_select == SwinModelSelect.SWIN_V2_S:
            if weights is not None and not isinstance(weights, Swin_V2_S_Weights):
                raise ValueError(f"Expected weights to be of type Swin_V2_S_Weights, got {type(weights)}")
            self.model = swin_v2_s(weights=weights)
        elif model_select == SwinModelSelect.SWIN_V2_B:
            if weights is not None and not isinstance(weights, Swin_V2_B_Weights):
                raise ValueError(f"Expected weights to be of type Swin_V2_B_Weights, got {type(weights)}")
            self.model = swin_v2_b(weights=weights)
        
        incoming_signal_size = self.model.head.in_features 
        self.model.head = nn.Linear(incoming_signal_size, num_classes)

    def forward(self, inputs, targets=None):
        
        _, _, h, w = inputs.shape
    
        if h % 4 != 0 or w % 4 != 0:
            raise ValueError(f"Input size must be divisible by 4, got {h}x{w}")
        logits = self.model(inputs)
        
        # Using dim=1 because dim 0 is the batch size
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(logits, dim=1)
        
        return {
            'logits': logits,
            'probs': probs,
            'labels': labels,
        }

    def compute_loss(self, outputs, targets):
        logits = outputs["logits"]
        labels = targets["label"]
        loss = f.cross_entropy(logits, labels)
        return {"loss": loss}
    

In [4]:
resize = Resize(224, 224)

dataset_train = ImageNetClassificationDataset(root_dir="/home/local/private/data/gtsrb_out", transform=resize)
dataset_val = ImageNetClassificationDataset(root_dir="/home/local/private/data/gtsrb_out", split="val", transform=resize)
print("Number of training samples:", len(dataset_train))
print("Number of validation samples:", len(dataset_val))

Number of training samples: 35347
Number of validation samples: 12630


In [5]:
data_loader_train = SimpleDataLoader(dataset=dataset_train, batch_size=32, shuffle=True)
data_loader_val = SimpleDataLoader(dataset=dataset_val, batch_size=32, shuffle=False)

In [ ]:
model = ResNet50(num_classes=dataset_train.get_num_classes(), weights=ResNet50_Weights.IMAGENET1K_V2)
adamW_optimizer = AdamW(model.parameters(), lr=0.001)
sgd_optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)
trainer = WandbTrainer(optimizer=adamW_optimizer,project="cv-course", run_name="test-resnet50-mnist")

: 

In [ ]:
trainer.fit(model=model, train_loader=data_loader_train, val_loader=data_loader_val)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/local/.netrc.
wandb: Currently logged in as: moritz-zideck (moritz-zideck-tu-wien) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/10
